### Imports

In [1]:
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from src.config import CONFIG
from src.lstm_utils.lstm_dataset import LSTMDataset
from src.lstm_utils.lstm_tokeniser import LSTMTokeniser
from src.models.lstm_classifier import LSTMClassifier

### Constants

In [2]:
TEST_DATA_PATH = Path(f'../data/test.csv')
OUTPUT_PATH = Path('../output')
OUTPUT_FILE_NAME = 'Group_40_B.csv'

MODEL_PATH = Path(f'../models/lstm.pt')

### Device

In [3]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps' if torch.backends.mps.is_available() else
    'cpu'
)
print(f'Using device: {device}')

Using device: mps


### Run demo

In [4]:
def run_demo() -> None:
    lstm_tokeniser = LSTMTokeniser()
    test_dataset = LSTMDataset(TEST_DATA_PATH, lstm_tokeniser, evaluate=True)
    test_dataloader = DataLoader(test_dataset, batch_size=CONFIG.batch_size, shuffle=False, num_workers=4, pin_memory=True)

    model = LSTMClassifier(lstm_tokeniser, dropout=0.5).to(device)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device), strict=False)
    model.eval()

    print(f'Running LSTM demo on {TEST_DATA_PATH}...')
    all_preds = []
    with torch.inference_mode():
        for batch in tqdm(test_dataloader, desc='Evaluating', unit='batches'):
            premise_ids = batch['premise_ids'].to(device)
            premise_mask = batch['premise_mask'].to(device)
            hypothesis_ids = batch['hypothesis_ids'].to(device)
            hypothesis_mask = batch['hypothesis_mask'].to(device)

            logits = model(premise_ids, premise_mask, hypothesis_ids, hypothesis_mask)
            preds = logits.argmax(dim=1)

            all_preds.extend(preds.cpu().tolist())

    # Verify no results are missing
    assert len(all_preds) == len(test_dataset)
    result_df = pd.DataFrame({'prediction': all_preds})

    # Write to a csv file
    print(f'Done. Writing results to {OUTPUT_PATH / OUTPUT_FILE_NAME}.')
    OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
    result_df.to_csv(OUTPUT_PATH / OUTPUT_FILE_NAME, index=False)

run_demo()

Running LSTM demo on ../data/test.csv...


Evaluating:   0%|          | 0/13 [00:00<?, ?batches/s]/Users/jackrong/University/34812-cwk-S-Project/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Evaluating: 100%|██████████| 13/13 [00:07<00:00,  1.78batches/s]


Done. Writing results to ../output/Group_40_B.csv.
